### Pré-traitement

In [13]:
# Import des librairies
import pandas as pd
import numpy as np
import hashlib

# On récupère les données de stats
data_ids = pd.read_csv("../data/entity_resolution.csv")
data_opta_analyst = pd.read_csv("../data/silver_analyst.csv")
data_fotmob = pd.read_csv("../data/fotmob_v2.csv")
data_sofascore = pd.read_csv("../data/sofascore_v2.csv")
data_understat = pd.read_csv("../data/silver_understat.csv")


In [14]:
# On crée l'identifiant issu d'opta analyst
def create_player_id(analyst_name):
    return hashlib.sha256(str(analyst_name).encode("utf-8")).hexdigest()[:16]

data_ids["opta_id"] = data_ids["analyst_name"].apply(create_player_id)

In [15]:
data_ids

,analyst_name,best_tm_id,best_method,fuzzy_score,reep_id,fotmob_id,sofascore_id,fbref_id,understat_id,opta_id
0,Aaron Greene,122407.0,reep,NaN,reep_pf7029e95,NaN,NaN,NaN,NaN,41e1a6833619b240
1,Aarón Escandell,284430.0,reep,NaN,reep_pbc6fe046,534955.0,368922.0,67669ce7,NaN,be5c61f30246a2b9
2,Aarón Herrera,401362.0,reep,NaN,reep_pcdb31119,NaN,NaN,d86e3070,NaN,bf8da63c2c534563
3,Aarón Martín,251878.0,reep,NaN,reep_pe0bb5f2e,684981.0,797286.0,2f3e911a,NaN,17b08b01696a4b12
4,Abakar Sylla,962555.0,reep,NaN,reep_peeaa665d,1359613.0,1170197.0,NaN,NaN,1b43e3b148b564e9
...,...,...,...,...,...,...,...,...,...,...
8636,Zachary Athekame,990637.0,reep,NaN,reep_p5406864a,1595629.0,1409700.0,NaN,NaN,2b627ddf55fdb08b
8637,Zander Clark,98067.0,reep,NaN,reep_p2b342b28,NaN,556366.0,d7fc839a,NaN,fce47f1f77818a44
8638,Óscar Trejo,30321.0,reep,NaN,reep_pab0700e3,21414.0,21949.0,fc647b34,NaN,5bd027a5f1bf4062
8639,Óscar Valentín,517753.0,reep,NaN,reep_p9d45ac50,956622.0,900008.0,592f3158,NaN,9486ca987c85e01a


In [16]:
# On comptabilise le nombre d'identifiants disponible pour un joueur sur les différents fournisseurs de données

cols_ids = ["best_tm_id","fotmob_id","sofascore_id","opta_id"]

n_complet = data_ids[cols_ids].notna().all(axis=1).sum()

print(n_complet)

n_total = len(data_ids)
pct_complet = n_complet / n_total * 100

print(f"{n_complet} lignes sur {n_total} ({pct_complet:.1f} %)")

1957
1957 lignes sur 8641 (22.6 %)


### Opta

In [17]:
# On ajoute l'identifiant d'opta
data_opta_analyst = data_opta_analyst.merge(
    data_ids[["analyst_name", "opta_id"]],left_on="name",
    right_on="analyst_name",how="left")

# Suppression de la colonne analyst_name ajoutée par le merge
data_opta_analyst = data_opta_analyst.drop(columns="analyst_name")

# On garde uniquement les lignes ayant opta_id
data_opta_analyst = data_opta_analyst[
    data_opta_analyst["opta_id"].notna()].copy()

# Variables à sommer

sum_cols = ["apps","minutes","atk_goals","atk_xg","atk_goals_vs_xg","atk_shots","atk_shots_on_target","def_tackles","def_interceptions",
            "def_possession_won","def_blocks","def_clearances","def_ground_duels_total","def_ground_duels_won","def_aerial_duels_total",
            "def_aerial_duels_won","pass_total","pass_open_play_total","pass_final_third","pass_crosses","pass_long_total",
            "pass_through_balls","carry_all_carries","carry_progressive","carry_distance_m","carry_prog_distance_m","carry_lead_to_shot",
            "carry_lead_to_goal","carry_lead_to_chance","carry_lead_to_assist","gk_goals_conceded","gk_saves","gk_xgot_conceded",
            "gk_goals_prevented"]

# Variables à moyenner en fonction des minutes

weighted_cols = ["atk_conversion_pct","atk_xg_per_shot","def_ground_duels_pct","def_aerial_duels_pct","pass_accuracy_pct",
    "pass_open_play_pct","pass_long_pct","carry_avg_distance_m","carry_prog_avg_distance_m","gk_save_pct"]

# Variables pour lesquelles on créera un per90

per90_cols = ["atk_goals","atp_xg","atk_shots","atk_shots_on_target","def_tackles","def_interceptions","def_possession_won",
    "def_blocks","def_clearances","def_ground_duels_total","def_ground_duels_won","def_aerial_duels_total","def_aerial_duels_won",
    "pass_total","pass_open_play_total","pass_final_third","pass_crosses","pass_long_total","pass_through_balls","carry_all_carries",
    "carry_progressive","carry_distance_m","carry_prog_distance_m","carry_lead_to_shot","carry_lead_to_goal","carry_lead_to_chance",
    "carry_lead_to_assist","gk_goals_conceded","gk_saves","gk_xgot_conceded","gk_goals_prevented"]

# On ne garde que les colonnes réellement présentes
sum_cols = [col for col in sum_cols if col in data_opta_analyst.columns]
weighted_cols = [col for col in weighted_cols if col in data_opta_analyst.columns]
per90_cols = [col for col in per90_cols if col in data_opta_analyst.columns]


def aggregate_player_opta(group):

    result = {}

    # opta_id
    result["opta_id"] = group.name

    # League : on concatène uniquement les valeurs différentes
    result["league"] = ", ".join(
        group["league"]
        .dropna()
        .astype(str)
        .drop_duplicates()
    )

    # Sommes
    for col in sum_cols:
        result[col] = group[col].sum(min_count=1)

    # Moyennes pondérées par les minutes

    for col in weighted_cols:

        mask = (group[col].notna() & group["minutes"].notna() & (group["minutes"] > 0))

        if mask.any():
            result[col] = np.average(group.loc[mask, col],weights=group.loc[mask, "minutes"])
        else:
            result[col] = np.nan

    # Autres variables : première valeur disponible

    processed_cols = ({"opta_id", "league"} | set(sum_cols) | set(weighted_cols))

    other_cols = [col
        for col in group.columns
        if col not in processed_cols
    ]

    for col in other_cols:

        values = group[col].dropna()

        result[col] = (values.iloc[0]
            if len(values) > 0
            else np.nan
        )

    return pd.Series(result)

# Agrégation par opta_id

data_opta_analyst_agg = (data_opta_analyst
    .groupby("opta_id", dropna=False)
    .apply(aggregate_player_opta)
    .reset_index(drop=True))

# Création des variables /90

for col in per90_cols:

    data_opta_analyst_agg[f"{col}_per90"] = np.where(data_opta_analyst_agg["minutes"] > 0,
        data_opta_analyst_agg[col] / data_opta_analyst_agg["minutes"] * 90, np.nan)

# Mettre name en première colonne
cols = ["name"] + [
    col for col in data_opta_analyst_agg.columns
    if col != "name"
]

data_opta_analyst_agg = data_opta_analyst_agg[cols]

In [18]:
data_opta_analyst_agg

,name,opta_id,league,apps,minutes,atk_goals,atk_xg,atk_goals_vs_xg,atk_shots,atk_shots_on_target,...,carry_distance_m_per90,carry_prog_distance_m_per90,carry_lead_to_shot_per90,carry_lead_to_goal_per90,carry_lead_to_chance_per90,carry_lead_to_assist_per90,gk_goals_conceded_per90,gk_saves_per90,gk_xgot_conceded_per90,gk_goals_prevented_per90
0,Jizz Hornkamp,0006ed1db043db39,Conference League,0,0,0.0,0.00,0.00,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Gianluigi Donnarumma,000b408e096a7ad7,"Premier League, Champions League",43,3870,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.953488,2.302326,1.111628,0.158140
2,Lorenzo Pellegrini,000c9c4ad406cee3,"Serie A, Europa League",33,1988,7.0,6.96,0.04,47.0,20.0,...,116.827968,45.235412,0.181087,0.0,0.316901,0.0,NaN,NaN,NaN,NaN
3,Ali Houary,001cec567d8d4cca,La Liga,2,125,0.0,0.07,-0.07,2.0,0.0,...,148.392000,88.128000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN
4,Álvaro Valles,00235a13fa36141b,"La Liga, Europa League",33,2970,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.151515,2.909091,1.093939,-0.027273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8636,Yvan Zaddy,ffd992942405c140,Ligue 1,1,1,0.0,0.00,0.00,0.0,0.0,...,1242.000000,1143.000000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN
8637,Kasper Boogaard,ffdb62673ef1a5c9,Conference League,4,211,0.0,0.27,-0.27,2.0,1.0,...,149.033175,72.298578,0.000000,0.0,0.426540,0.0,NaN,NaN,NaN,NaN
8638,Rico Lewis,fff0dbc42a457bd9,"Premier League, Champions League",15,620,0.0,0.36,-0.36,5.0,0.0,...,156.004839,79.824194,0.000000,0.0,0.145161,0.0,NaN,NaN,NaN,NaN
8639,Christian Burgess,fff2784d04a65435,Champions League,8,720,0.0,0.09,-0.09,2.0,1.0,...,61.975000,20.275000,0.000000,0.0,0.000000,0.0,NaN,NaN,NaN,NaN


In [20]:
data_opta_analyst_agg.to_csv("data_opta_analyst_agg.csv",index=False)

### Sofascore

### Fotmob